In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier
import joblib

In [ ]:
df=pd.read_csv("/content/heart_disease.csv")

In [ ]:
df.columns

Index(['Age', 'Gender', 'Blood Pressure', 'Cholesterol Level',
       'Exercise Habits', 'Smoking', 'Family Heart Disease', 'Diabetes', 'BMI',
       'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol',
       'Alcohol Consumption', 'Stress Level', 'Sleep Hours',
       'Sugar Consumption', 'Triglyceride Level', 'Fasting Blood Sugar',
       'CRP Level', 'Homocysteine Level', 'Heart Disease Status'],
      dtype='object')

In [ ]:
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Age                   9971 non-null   float64
 1   Gender                9981 non-null   object 
 2   Blood Pressure        9981 non-null   float64
 3   Cholesterol Level     9970 non-null   float64
 4   Exercise Habits       9975 non-null   object 
 5   Smoking               9975 non-null   object 
 6   Family Heart Disease  9979 non-null   object 
 7   Diabetes              9970 non-null   object 
 8   BMI                   9978 non-null   float64
 9   High Blood Pressure   9974 non-null   object 
 10  Low HDL Cholesterol   9975 non-null   object 
 11  High LDL Cholesterol  9974 non-null   object 
 12  Alcohol Consumption   7414 non-null   object 
 13  Stress Level          9978 non-null   object 
 14  Sleep Hours           9975 non-null   float64
 15  Sugar Consumption   

,Age,Blood Pressure,Cholesterol Level,BMI,Sleep Hours,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level
count,9971.000000,9981.000000,9970.000000,9978.000000,9975.000000,9974.000000,9978.000000,9974.000000,9980.000000
mean,49.296259,149.757740,225.425577,29.077269,6.991329,250.734409,120.142213,7.472201,12.456271
std,18.193970,17.572969,43.575809,6.307098,1.753195,87.067226,23.584011,4.340248,4.323426
min,18.000000,120.000000,150.000000,18.002837,4.000605,100.000000,80.000000,0.003647,5.000236
25%,34.000000,134.000000,187.000000,23.658075,5.449866,176.000000,99.000000,3.674126,8.723334
50%,49.000000,150.000000,226.000000,29.079492,7.003252,250.000000,120.000000,7.472164,12.409395
75%,65.000000,165.000000,263.000000,34.520015,8.531577,326.000000,141.000000,11.255592,16.140564
max,80.000000,180.000000,300.000000,39.996954,9.999952,400.000000,160.000000,14.997087,19.999037


In [ ]:
df.columns

Index(['Age', 'Gender', 'Blood Pressure', 'Cholesterol Level',
       'Exercise Habits', 'Smoking', 'Family Heart Disease', 'Diabetes', 'BMI',
       'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol',
       'Alcohol Consumption', 'Stress Level', 'Sleep Hours',
       'Sugar Consumption', 'Triglyceride Level', 'Fasting Blood Sugar',
       'CRP Level', 'Homocysteine Level', 'Heart Disease Status'],
      dtype='object')

In [ ]:
df.sample(5)

,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,...,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
6307,68.0,Male,121.0,220.0,High,No,No,Yes,33.194491,No,...,No,NaN,High,6.674897,High,204.0,104.0,10.242545,6.505413,No
6222,32.0,Male,177.0,284.0,Low,No,No,Yes,27.087980,No,...,No,High,Medium,4.181072,Medium,381.0,151.0,9.373252,7.872450,No
3404,50.0,Female,151.0,200.0,High,No,Yes,No,23.730447,Yes,...,Yes,NaN,Medium,8.406058,High,243.0,129.0,13.655935,19.476041,No
9839,53.0,Female,162.0,194.0,High,Yes,No,Yes,34.895305,Yes,...,No,Low,Low,4.419077,Medium,159.0,128.0,12.949735,6.357986,Yes
7184,47.0,Female,140.0,248.0,Low,Yes,Yes,Yes,39.183354,Yes,...,Yes,Medium,High,7.383420,High,263.0,133.0,6.252267,10.015691,No


In [ ]:
(df.isnull().sum() / len(df)) * 100

,0
Age,0.29
Gender,0.19
Blood Pressure,0.19
Cholesterol Level,0.30
Exercise Habits,0.25
Smoking,0.25
Family Heart Disease,0.21
Diabetes,0.30
BMI,0.22
High Blood Pressure,0.26


In [ ]:
df['Heart Disease Status'] = df['Heart Disease Status'].map({ 'Yes': 1,  'No': 0})

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

In [ ]:
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

/tmp/ipykernel_2909/4074283006.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [ ]:
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

/tmp/ipykernel_2909/1925696021.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
df.isnull().sum()

,0
Age,0
Gender,0
Blood Pressure,0
Cholesterol Level,0
Exercise Habits,0
Smoking,0
Family Heart Disease,0
Diabetes,0
BMI,0
High Blood Pressure,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
disease_percent = df['Heart Disease Status'].value_counts(normalize=True) * 100
print(disease_percent)

Heart Disease Status
0    80.0
1    20.0
Name: proportion, dtype: float64


In [ ]:
#Encoding categorical features
encoder = OrdinalEncoder()
df[['Gender','Exercise Habits', 'Smoking', 'Family Heart Disease', 'Diabetes',
       'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol',
       'Alcohol Consumption', 'Stress Level','Sugar Consumption']] = encoder.fit_transform(df[['Gender','Exercise Habits', 'Smoking', 'Family Heart Disease', 'Diabetes',
       'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol',
       'Alcohol Consumption', 'Stress Level','Sugar Consumption']])

In [ ]:
df.dtypes

,0
Age,float64
Gender,float64
Blood Pressure,float64
Cholesterol Level,float64
Exercise Habits,float64
Smoking,float64
Family Heart Disease,float64
Diabetes,float64
BMI,float64
High Blood Pressure,float64


In [ ]:
X = df.drop('Heart Disease Status', axis=1)
y = df['Heart Disease Status']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

xgb = XGBClassifier(
     n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    scale_pos_weight=4,
    random_state=42
)

xgb.fit(X_train.values, y_train)

y_prob = xgb.predict_proba(X_test.values)[:,1]
threshold = 0.45


y_pred = (y_prob > threshold).astype(int)

print("XGBoost")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

XGBoost
Accuracy: 0.496
              precision    recall  f1-score   support

           0       0.80      0.50      0.61      1613
           1       0.19      0.48      0.27       387

    accuracy                           0.50      2000
   macro avg       0.49      0.49      0.44      2000
weighted avg       0.68      0.50      0.55      2000



In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score

# Generate classification report as dictionary
report = classification_report(y_test, y_pred, output_dict=True)

# Extract only class 1 metrics
class1_metrics = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [
        round(accuracy_score(y_test, y_pred), 4),
        round(report['1']['precision'], 4),
        round(report['1']['recall'], 4),
        round(report['1']['f1-score'], 4)
    ]
}

# Convert to DataFrame
table = pd.DataFrame(class1_metrics)

print("XGBoost - Class 1 Performance")
print(table)

report_df.style.set_caption("Classification Report - XGBoost")\
.set_table_styles([
    {'selector':'th','props':[('border','1px solid black'),('text-align','center')]},
    {'selector':'td','props':[('border','1px solid black'),('text-align','center')]}
])

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(cm,
                     index=["Actual No Disease","Actual Heart Disease"],
                     columns=["Predicted No Disease","Predicted Heart Disease"])

print("Confusion Matrix")
print(cm_df)

cm_df.style.set_caption("Confusion Matrix - XGBoost")\
.set_table_styles([
    {'selector':'th','props':[('border','1px solid black'),('text-align','center')]},
    {'selector':'td','props':[('border','1px solid black'),('text-align','center')]}
])

XGBoost - Class 1 Performance
      Metric   Score
0   Accuracy  0.4960
1  Precision  0.1879
2     Recall  0.4832
3   F1 Score  0.2706
Confusion Matrix
                      Predicted No Disease  Predicted Heart Disease
Actual No Disease                      805                      808
Actual Heart Disease                   200                      187


,Predicted No Disease,Predicted Heart Disease
Actual No Disease,805,808
Actual Heart Disease,200,187


In [ ]:
joblib.dump(xgb, "heart_model.pkl")

['heart_model.pkl']

In [ ]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.3 MB/s eta 0:00:00
